In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

# Siapkan list untuk menampung hasil
comparison_data = []

print("Sedang menghitung detail metrik...")

for name, model in models.items():
    # Prediksi
    y_pred = model.predict(X_test)

    # Hitung metrics (weighted rata-rata memperhitungkan jumlah data per kelas)
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

    comparison_data.append({
        "Algoritma": name,
        "Akurasi": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

# Buat DataFrame
df_comparison = pd.DataFrame(comparison_data)

# Tampilkan Tabel (Copy tabel ini ke Jurnal Anda)
print("\n--- TABEL PERBANDINGAN PERFORMA ---")
print(df_comparison.round(4)) # Bulatkan 4 angka belakang koma

In [ ]:
from sklearn.model_selection import cross_val_score

print("--- MENJALANKAN 10-FOLD CROSS VALIDATION ---")
print("(Proses ini mungkin memakan waktu 1-2 menit)\n")

cv_results = []

for name, model in models.items():
    # Lakukan 10 kali pengujian (cv=10)
    scores = cross_val_score(model, X, y, cv=10, scoring='accuracy')

    print(f"{name}: Rata-rata Akurasi = {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)")

    cv_results.append({
        "Algoritma": name,
        "Mean Accuracy": scores.mean(),
        "Std Dev": scores.std()
    })

# Visualisasi Cross Validation
df_cv = pd.DataFrame(cv_results)
plt.figure(figsize=(8,5))
sns.barplot(x='Algoritma', y='Mean Accuracy', data=df_cv, palette='magma')
plt.ylim(0, 1.0)
plt.title('Rata-rata Akurasi dengan 10-Fold Cross Validation')
plt.ylabel('Rata-rata Akurasi')
plt.show()

In [ ]:
from wordcloud import WordCloud

# Pastikan Anda punya data teks mentah dan labelnya
# Kita ambil dari df (hasil Tahap 4)
df_viz = df.copy()

def generate_wordcloud(text_data, title):
    # Gabungkan semua teks jadi satu string panjang
    all_text = " ".join(text_data.astype(str))

    wordcloud = WordCloud(width=800, height=400,
                          background_color='white',
                          colormap='viridis',
                          max_words=200).generate(all_text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=15)
    plt.show()

# 1. Word Cloud Positif
print("Generating Word Cloud: POSITIF...")
text_positif = df_viz[df_viz['label_otomatis'] == 'Positif']['text_clean']
if len(text_positif) > 0:
    generate_wordcloud(text_positif, "Word Cloud - Sentimen Positif")

# 2. Word Cloud Negatif
print("Generating Word Cloud: NEGATIF...")
text_negatif = df_viz[df_viz['label_otomatis'] == 'Negatif']['text_clean']
if len(text_negatif) > 0:
    generate_wordcloud(text_negatif, "Word Cloud - Sentimen Negatif")

In [ ]:
# Pilih satu model untuk dicek (Misal Naive Bayes)
model_cek = nb_model
y_pred_cek = model_cek.predict(X_test)

# Kembalikan angka ke label teks
y_test_label = le.inverse_transform(y_test)
y_pred_label = le.inverse_transform(y_pred_cek)

# Buat DataFrame hasil prediksi
df_error = pd.DataFrame({
    'Text_Asli': X_test_raw if 'X_test_raw' in locals() else "Teks ter-enkripsi di TF-IDF",
    # Catatan: Karena X sudah jadi angka TF-IDF, kita susah baca teks aslinya di sini
    # kecuali kita simpan indexnya sebelum split.
    # Tapi kita bisa bandingkan labelnya saja:
    'Label_Asli': y_test_label,
    'Prediksi_Model': y_pred_label
})

# Filter yang SALAH saja
df_salah = df_error[df_error['Label_Asli'] != df_error['Prediksi_Model']]

print(f"Total Kesalahan Prediksi: {len(df_salah)} dari {len(y_test)} data uji.")
print("\n--- Contoh 5 Data yang Salah Diprediksi ---")
print(df_salah.head())

# TIPS:
# Jika output kolom 'Text_Asli' tidak muncul teksnya,
# itu wajar karena X_train/test sudah berupa angka TF-IDF.
# Analisisnya: "Model sering salah memprediksi Netral menjadi Negatif" (lihat pola tabel di atas).